# Rates derivatives: deposits, FRA, IRS

**Prerequisites:** **05**–**06**. We price **deposits** at two horizons, a **forward rate agreement**, and a vanilla **IRS**.


## Concept

- **Deposits:** PV from quoted rate vs discount factors.
- **FRA:** payoff linked to fixing vs strike forward rate.
- **IRS:** fixed leg vs projected floating leg.

**PV01** / **bucketed DV01** summarize parallel vs tenor-local rate sensitivities when registered.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (rates derivatives).")


## Instrument JSON

**IRS fixed leg:** The fixed rate is **~4.5%** to sit near **par** against the bundled OIS/SOFR snapshot; a **3%** coupon would be intentionally off-market (large PV from fixed–float mismatch) if you use it for illustration.


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

def deposit(mid: str, mat: str, rate: float) -> str:
    return json.dumps(
        {
            "type": "deposit",
            "spec": {
                "id": mid,
                "notional": {"amount": 1_000_000.0, "currency": "USD"},
                "start_date": AS_OF_STR,
                "maturity": mat,
                "day_count": "Act360",
                "quote_rate": rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        }
    )

dep_3m = deposit("DEP-3M", "2025-04-15", 0.0505)
dep_1y = deposit("DEP-1Y", "2026-01-15", 0.0490)

fra = json.dumps(
    {
        "type": "forward_rate_agreement",
        "spec": {
            "id": "FRA-3x6",
            "notional": {"amount": "10000000", "currency": "USD"},
            "start_date": "2025-04-15",
            "maturity": "2025-07-15",
            "fixed_rate": "0.048",
            "day_count": "Act360",
            "discount_curve_id": "USD-OIS",
            "forward_curve_id": "USD-SOFR-3M",
            "fixing_date": "2025-04-10",
            "reset_lag": 2,
            "side": "receive",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

irs = json.dumps(
    {
        "type": "interest_rate_swap",
        "spec": {
            "id": "IRS-5Y-USD",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "fixed": {
                "discount_curve_id": "USD-OIS",
                "rate": 0.045,
                "frequency": {"count": 6, "unit": "months"},
                "day_count": "Thirty360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "start": "2025-04-15",
                "end": "2030-04-15",
                "par_method": None,
                "compounding_simple": True,
            },
            "float": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": 0.0,
                "frequency": {"count": 3, "unit": "months"},
                "day_count": "Act360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "reset_lag_days": 2,
                "start": "2025-04-15",
                "end": "2030-04-15",
                "compounding": "Simple",
            },
            "attributes": {},
        },
    }
)

for label, js in (("dep_3m", dep_3m), ("dep_1y", dep_1y), ("fra", fra)):
    try:
        validate_instrument_json(js)
        print(label, "OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("market_json chars:", len(market_json))


## Pricing


In [ ]:
for label, js in (("dep_3m", dep_3m), ("dep_1y", dep_1y), ("fra", fra)):
    try:
        out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)

try:
    out = price_instrument_with_metrics(irs, market_json, AS_OF_STR, model="discounting")
    print("irs", ValuationResult.from_json(out))
except Exception as exc:
    print("irs", type(exc).__name__, exc)


## Metrics


In [ ]:
for label, js in (("dep_3m", dep_3m), ("irs", irs)):
    try:
        out = price_instrument_with_metrics(
            js, market_json, AS_OF_STR, model="discounting", metrics=["dv01", "bucketed_dv01", "npv01"]
        )
        vr = ValuationResult.from_json(out)
        print("—", label, "dv01:", vr.get_metric("dv01"))
        b = vr.get_metric("bucketed_dv01")
        if b is not None:
            print("  bucketed_dv01:", b)
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)
print("PV01 / bucketed DV01: interpret as parallel vs key-rate style sensitivities when present.")


## Takeaways

- **Deposits** anchor short-end PV; **FRAs** and **swaps** need consistent **forward** and **discount** IDs.
- **Unseasoned** swaps avoid missing historical fixings before `as_of`.
- Request **`bucketed_dv01`** when you want tenor-bucket narratives; omit silently if not registered.
